<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/AE%20Multinomial%20Logistic%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '/content/136552_136553_A_B_RECORDS_AI_ML_AUGMENTED.xlsx'
df = pd.read_excel(file_path)

# Display basic info and first few rows
print("Dataset Info:")
display(df.info())
print("\nFirst 5 rows:")
display(df.head())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column                                                               Non-Null Count  Dtype 
---  ------                                                               --------------  ----- 
 0   Aircraft Engine Predictive Maintenance Dataset Augmentation Summary  18 non-null     object
 1   Unnamed: 1                                                           18 non-null     object
 2   Unnamed: 2                                                           14 non-null     object
 3   Unnamed: 3                                                           6 non-null      object
 4   Unnamed: 4                                                           9 non-null      object
 5   Unnamed: 5                                                           9 non-null      object
dtypes: object(6)
memory usage: 1.3+ KB


None


First 5 rows:


,Aircraft Engine Predictive Maintenance Dataset Augmentation Summary,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Purpose,AI/ML model development for predictive mainten...,NaN,NaN,NaN,NaN
2,Important limitation,"The 2,000 added records are synthetic analytic...",NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,Sheet,Original Records,Synthetic Fault Records,Total Records,Engine Serial,Record A/B


In [ ]:
# Check for missing values and unique values in potential target columns
print("Missing values per column:")
display(df.isnull().sum())

# Assuming we need to find a categorical target for Multinomial Logistic Regression
# We will list columns with fewer than 20 unique values as potential targets
print("\nPotential categorical targets (unique value counts):")
for col in df.select_dtypes(include=['object', 'category', 'int']).columns:
    unique_count = df[col].nunique()
    if 2 < unique_count < 20:
        print(f"{col}: {unique_count} unique values")

Missing values per column:


,0
Aircraft Engine Predictive Maintenance Dataset Augmentation Summary,7
Unnamed: 1,7
Unnamed: 2,11
Unnamed: 3,19
Unnamed: 4,16
Unnamed: 5,16



Potential categorical targets (unique value counts):
Aircraft Engine Predictive Maintenance Dataset Augmentation Summary: 18 unique values
Unnamed: 1: 17 unique values
Unnamed: 2: 11 unique values
Unnamed: 3: 5 unique values
Unnamed: 4: 7 unique values
Unnamed: 5: 7 unique values


In [ ]:
import pandas as pd

# Inspect available sheets to find the actual data
xl = pd.ExcelFile('/content/136552_136553_A_B_RECORDS_AI_ML_AUGMENTED.xlsx')
print(f'Sheets found: {xl.sheet_names}')

# Load the primary data sheet (assuming it is the second sheet or has a relevant name)
target_sheet = xl.sheet_names[1] if len(xl.sheet_names) > 1 else xl.sheet_names[0]
df_main = pd.read_excel('/content/136552_136553_A_B_RECORDS_AI_ML_AUGMENTED.xlsx', sheet_name=target_sheet)

print(f'\nLoaded sheet: {target_sheet}')
display(df_main.head())

Sheets found: ['AI_ML_SUMMARY', '136552_INCIDENT_RECORD_A', '136552_INCIDENT_RECORD_B', '136553_INCIDENT_RECORD_A', '136553_INCIDENT_RECORD_B']

Loaded sheet: 136552_INCIDENT_RECORD_A


,SOURCE,ECU Operating Time,Leg Number,N1,N2,EGT,ITT Display,ECU TT2,ECU PS,CGV position,...,Incident Type,Engine Serial,Record Origin,Synthetic Fault Category,Fault Record Description,Fault Severity,Target Spare Part,Suggested Maintenance Action,ML Fault Label,Synthetic Record ID
0,SOURCE_1,21997812.9,2740,52.25,79.31,386.75,484.38,14.19,13.79,42.11,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
1,SOURCE_1,21997814.9,2740,50.77,79.36,394.19,484.38,14.16,13.80,41.72,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
2,SOURCE_1,21997816.9,2740,50.61,79.30,395.94,486.38,14.09,13.81,41.81,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
3,SOURCE_1,21997818.9,2740,50.56,79.28,396.00,486.25,14.03,13.82,41.81,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
4,SOURCE_1,21997820.9,2740,49.94,78.61,394.56,485.00,14.02,13.83,42.79,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Combine all incident record sheets
sheets = ['136552_INCIDENT_RECORD_A', '136552_INCIDENT_RECORD_B', '136553_INCIDENT_RECORD_A', '136553_INCIDENT_RECORD_B']
df_list = [pd.read_excel(file_path, sheet_name=s) for s in sheets]
all_data = pd.concat(df_list, ignore_index=True)

# Preprocessing: Select numerical features and target
target_col = 'ML Fault Label'
# Drop metadata and identify numerical columns
exclude_cols = ['ECU Operating Time', 'Leg Number', 'Engine Serial', 'Synthetic Record ID']
X = all_data.select_dtypes(include=[np.number]).drop(columns=exclude_cols, errors='ignore')
y = all_data[target_col]

# FIX: Instead of dropping all rows with NaNs (which resulted in 0 samples),
# we drop columns that are entirely empty and fill remaining NaNs with the mean.
X = X.dropna(axis=1, how='all')
X = X.fillna(X.mean())

# Ensure y matches X after any potential cleaning
valid_indices = X.index
y = y.loc[valid_indices]

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y.astype(str))

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Multinomial Logistic Regression
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluation
y_pred = model.predict(X_test_scaled)
print(f'Model Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Model Accuracy: 0.9805

Classification Report:
                         precision    recall  f1-score   support

FAULT_BEARING_VIBRATION       0.97      1.00      0.99        70
       FAULT_COMPRESSOR       0.76      0.75      0.75        55
    FAULT_FUEL_METERING       0.81      0.83      0.82        65
      FAULT_HOT_SECTION       0.91      0.90      0.90        68
      FAULT_LUBRICATION       1.00      0.97      0.98        65
FAULT_OVERSPEED_CONTROL       1.00      1.00      1.00        40
  FAULT_SENSOR_ACTUATOR       1.00      1.00      1.00        37
      OBSERVED_INCIDENT       1.00      1.00      1.00      1341

               accuracy                           0.98      1741
              macro avg       0.93      0.93      0.93      1741
           weighted avg       0.98      0.98      0.98      1741



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Combine all incident sheets into one DataFrame
sheets = ['136552_INCIDENT_RECORD_A', '136552_INCIDENT_RECORD_B', '136553_INCIDENT_RECORD_A', '136553_INCIDENT_RECORD_B']
df_list = [pd.read_excel(file_path, sheet_name=s) for s in sheets]
all_data = pd.concat(df_list, ignore_index=True)

# Identify numerical features and target
target_col = 'ML Fault Label'
exclude_cols = ['ECU Operating Time', 'Leg Number', 'Engine Serial', 'Synthetic Record ID']
X = all_data.select_dtypes(include=[np.number]).drop(columns=exclude_cols, errors='ignore')
y = all_data[target_col]

# FIX: Instead of dropping all rows with NaNs (which resulted in 0 samples),
# we drop columns that are entirely empty and fill remaining NaNs with the mean.
X = X.dropna(axis=1, how='all')
X = X.fillna(X.mean())

# Ensure y matches X after any potential cleaning
y = y.loc[X.index]

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y.astype(str))

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Multinomial Logistic Regression
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict and Evaluate
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.9805

Classification Report:
                         precision    recall  f1-score   support

FAULT_BEARING_VIBRATION       0.97      1.00      0.99        70
       FAULT_COMPRESSOR       0.76      0.75      0.75        55
    FAULT_FUEL_METERING       0.81      0.83      0.82        65
      FAULT_HOT_SECTION       0.91      0.90      0.90        68
      FAULT_LUBRICATION       1.00      0.97      0.98        65
FAULT_OVERSPEED_CONTROL       1.00      1.00      1.00        40
  FAULT_SENSOR_ACTUATOR       1.00      1.00      1.00        37
      OBSERVED_INCIDENT       1.00      1.00      1.00      1341

               accuracy                           0.98      1741
              macro avg       0.93      0.93      0.93      1741
           weighted avg       0.98      0.98      0.98      1741



In [ ]:
import pickle

# Save the model to a file
model_filename = 'multinomial_logistic_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

# Save the label encoder as well to decode future predictions
le_filename = 'label_encoder.pkl'
with open(le_filename, 'wb') as file:
    pickle.dump(le, file)

# Save the scaler to ensure future data is scaled the same way
scaler_filename = 'scaler.pkl'
with open(scaler_filename, 'wb') as file:
    pickle.dump(scaler, file)

print(f"Model, Label Encoder, and Scaler saved successfully as .pkl files.")

Model, Label Encoder, and Scaler saved successfully as .pkl files.


In [ ]:
# Check sheet names in the Excel file
xl = pd.ExcelFile(file_path)
print(f"Sheet names: {xl.sheet_names}")

Sheet names: ['AI_ML_SUMMARY', '136552_INCIDENT_RECORD_A', '136552_INCIDENT_RECORD_B', '136553_INCIDENT_RECORD_A', '136553_INCIDENT_RECORD_B']


In [ ]:
# Assuming the data is in the second sheet based on the summary description
# We will try to load the 'Original Records' or 'Synthetic Fault Records' if they exist as sheets
# Otherwise, we'll load the second sheet by index.
sheet_to_load = xl.sheet_names[1] if len(xl.sheet_names) > 1 else xl.sheet_names[0]
df_data = pd.read_excel(file_path, sheet_name=sheet_to_load)

print(f"\nLoaded sheet: {sheet_to_load}")
print("Data shape:", df_data.shape)
display(df_data.head())


Loaded sheet: 136552_INCIDENT_RECORD_A
Data shape: (1861, 54)


,SOURCE,ECU Operating Time,Leg Number,N1,N2,EGT,ITT Display,ECU TT2,ECU PS,CGV position,...,Incident Type,Engine Serial,Record Origin,Synthetic Fault Category,Fault Record Description,Fault Severity,Target Spare Part,Suggested Maintenance Action,ML Fault Label,Synthetic Record ID
0,SOURCE_1,21997812.9,2740,52.25,79.31,386.75,484.38,14.19,13.79,42.11,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
1,SOURCE_1,21997814.9,2740,50.77,79.36,394.19,484.38,14.16,13.80,41.72,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
2,SOURCE_1,21997816.9,2740,50.61,79.30,395.94,486.38,14.09,13.81,41.81,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
3,SOURCE_1,21997818.9,2740,50.56,79.28,396.00,486.25,14.03,13.82,41.81,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN
4,SOURCE_1,21997820.9,2740,49.94,78.61,394.56,485.00,14.02,13.83,42.79,...,INCIDENT_RECORD_A | Event Trigger 0,136552,ORIGINAL,Observed incident record,Original source record retained without altera...,Observed,To be determined from maintenance records,Use approved maintenance documentation and eng...,OBSERVED_INCIDENT,NaN


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Combine all relevant data sheets
sheets = ['136552_INCIDENT_RECORD_A', '136552_INCIDENT_RECORD_B', '136553_INCIDENT_RECORD_A', '136553_INCIDENT_RECORD_B']
df_list = []
for s in sheets:
    try:
        temp_df = pd.read_excel(file_path, sheet_name=s)
        df_list.append(temp_df)
    except Exception as e:
        print(f'Could not load sheet {s}: {e}')

all_data = pd.concat(df_list, ignore_index=True)

# Data Cleaning: Select numerical features and the target label
target_col = 'ML Fault Label'
# Dropping metadata columns that aren't sensor readings
exclude_cols = ['ECU Operating Time', 'Leg Number', 'Engine Serial', 'Synthetic Record ID']
X = all_data.select_dtypes(include=[np.number]).drop(columns=exclude_cols, errors='ignore')
y = all_data[target_col]

# Handle missing values by dropping columns/rows with NaNs
X = X.dropna(axis=1, how='all')
valid_mask = X.notna().all(axis=1) & y.notna()
X = X[valid_mask]
y = y[valid_mask]

# Encode categorical target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Multinomial Logistic Regression
# 'multi_class' is set to 'multinomial' for the requested model type
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluation
y_pred = model.predict(X_test_scaled)
print(f'Model Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.